# VELNOR Core Prime — Qwen3-8B QLoRA, full curriculum + coding pack

Trains the **Core Prime** adapter: 397 examples = 138 embedded + 150 general-curriculum + 64 Core Prime persona/behaviour + 45 real-engineering coding rows.

Config: 2 epochs with early stopping (patience 1), seq_len 640 (covers the longest coding answer, max ~600 tokens), grad accum 8. Selection is by held-out loss, and the promotion gate blocks a regression.

Free GPU only. Preflight requires >= 14 GiB total and >= 12 GiB free VRAM, otherwise nothing starts.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU attached. Select a free GPU runtime; CPU training is disabled.')
p = torch.cuda.get_device_properties(0)
free, total = torch.cuda.mem_get_info(0)
print('GPU:', p.name, '| total GiB:', round(total/1024**3,2), '| free GiB:', round(free/1024**3,2))
if total < 14*1024**3 or free < 12*1024**3:
    raise RuntimeError('GPU does not meet the Qwen3-8B QLoRA preflight. No training started.')


In [ ]:
import subprocess
REPO = '/content/velnor-general'
subprocess.run(['rm', '-rf', REPO])
subprocess.run(['git', 'clone', '--branch', 'feat/qwen3-8b-qlora', '--depth', '1',
                'https://github.com/vectolith/velnor.git', REPO], check=True)
print(subprocess.check_output(['git', '-C', REPO, 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
%cd /content/velnor-general
!python -m pip install -q -r training/requirements-qwen3-qlora.txt
import torch
assert torch.cuda.is_available(), 'CUDA was lost during dependency installation.'
print('CUDA:', torch.version.cuda, '| PyTorch:', torch.__version__)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, subprocess
RUNS = Path('/content/drive/MyDrive/VELNOR-training-runs')
RUNS.mkdir(parents=True, exist_ok=True)
run_name = 'velnor-core-prime-e1-' + subprocess.check_output(['date','-u','+%Y%m%dT%H%M%SZ'], text=True).strip()
RUN_DIR = RUNS / run_name
OUT_DIR = RUN_DIR / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=False)
os.environ['VELNOR_QWEN3_OUTPUT'] = str(OUT_DIR)
os.environ.pop('VELNOR_QWEN3_RESUME', None)
os.environ.pop('VELNOR_QWEN3_PREV_BEST_LOSS', None)
os.environ['VELNOR_QWEN3_MAX_SEQ_LEN'] = '640'
os.environ['VELNOR_QWEN3_EPOCHS'] = '2'
os.environ['VELNOR_QWEN3_GRAD_ACCUM'] = '8'
os.environ['VELNOR_QWEN3_PATIENCE'] = '1'
print('run dir:', RUN_DIR)
print('seq_len 640 | epochs 2 | patience 1')


In [ ]:
%%bash
set -euo pipefail
cd /content/velnor-general/training
python -u train_qwen3_8b_qlora.py 2>&1 | tee "$VELNOR_QWEN3_OUTPUT/training.log"
test -s "$VELNOR_QWEN3_OUTPUT/adapter/adapter_config.json"
test -s "$VELNOR_QWEN3_OUTPUT/training_report.json"
echo TRAINING_DONE


In [ ]:
import json
report = json.loads((OUT_DIR / 'training_report.json').read_text(encoding='utf-8'))
print('status:', report['status'])
print('dataset:', report['dataset']['train_examples'], 'train /', report['dataset']['heldout_examples'], 'heldout')
print('method:', report['method']['epochs_run'], 'epoch(s), lr', report['method']['learning_rate'], 'seq', report['method']['max_sequence_length'])
print('validation:', report['validation']['pre_training_loss'], '->', report['validation']['best_loss'], 'best epoch', report['validation']['best_epoch'])
print('rubric base:', report['rubric']['base']['passed'], '/', report['rubric']['base']['total'])
print('rubric tuned:', report['rubric']['tuned']['passed'], '/', report['rubric']['tuned']['total'], 'delta', report['rubric']['delta_passed'])
print('gate:', json.dumps(report['promotion_gate']))
for d in report['rubric']['tuned']['by_domain']:
    print(' ', d)


In [ ]:
import shutil
archive = shutil.make_archive(str(RUN_DIR / 'velnor-qwen3-8b-general-e1-artifacts'), 'zip', root_dir=str(OUT_DIR))
print('archived:', archive)
print(RUN_DIR)
